# IPL Prediction — Data Collection & Preprocessing

Yeh notebook poora data pipeline explain karti hai:
1. Cricsheet se raw data kaise download karte hain
2. Raw CSVs ka structure
3. Preprocessing pipeline walkthrough
4. Har processed file ka analysis
5. Player stats & team form
6. Feature engineering summary

**Data source:** [cricsheet.org](https://cricsheet.org/) — open-source ball-by-ball cricket data

## 0. Setup & Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT        = Path('..').resolve()
RAW_DIR     = ROOT / 'data' / 'ipl_csv2'
PROC_DIR    = ROOT / 'data' / 'processed'
MODELS_DIR  = ROOT / 'models'

print('Project root :', ROOT)
print('Raw match files:', len(list(RAW_DIR.glob('*.csv'))) if RAW_DIR.exists() else 'Not found')
print('Processed files:', len(list(PROC_DIR.glob('*.csv'))) if PROC_DIR.exists() else 'Not found')

---
## 1. Data Collection — Cricsheet Download

Agar `data/ipl_csv2/` folder empty hai ya exist nahi karta, to neeche wale cell se download karo.
Otherwise skip kar sakte ho directly Section 2 pe.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA DOWNLOAD — run only if data/ipl_csv2/ is empty or missing
# ─────────────────────────────────────────────────────────────────────────────
import urllib.request, zipfile, io

CRICSHEET_IPL_URL = 'https://cricsheet.org/downloads/ipl_csv2.zip'

def download_cricsheet(url: str, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    existing = list(dest_dir.glob('*.csv'))
    if existing:
        print(f'Already have {len(existing)} CSV files in {dest_dir}. Skipping download.')
        return
    print(f'Downloading from {url} ...')
    with urllib.request.urlopen(url) as resp:
        data = resp.read()
    print(f'Downloaded {len(data)/1e6:.1f} MB. Extracting...')
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(dest_dir)
    n = len(list(dest_dir.glob('*.csv')))
    print(f'Extracted {n} files to {dest_dir}')

# Uncomment to download:
# download_cricsheet(CRICSHEET_IPL_URL, RAW_DIR)
print('Download cell ready (uncomment last line to run)')

---
## 2. Raw Data Structure

### 2.1 Match file pairs
Har match ke 2 files hote hain:
- `{match_id}.csv`       → ball-by-ball delivery data
- `{match_id}_info.csv`  → match metadata (teams, venue, toss, result)

In [ ]:
# Count match files
if RAW_DIR.exists():
    ball_files = sorted([f for f in RAW_DIR.glob('*.csv') if '_info' not in f.name])
    info_files = sorted(RAW_DIR.glob('*_info.csv'))
    print(f'Ball-by-ball files : {len(ball_files)}')
    print(f'Info files         : {len(info_files)}')
    print(f'\nSample match IDs   : {[f.stem for f in ball_files[:5]]}')
else:
    print('RAW_DIR not found. Download data first (Section 1).')

In [ ]:
# ── Load one sample ball-by-ball file ──────────────────────────────────────────
if RAW_DIR.exists() and ball_files:
    sample_match = ball_files[100]   # pick a mid-range match
    df_ball = pd.read_csv(sample_match)

    print(f'Match file : {sample_match.name}')
    print(f'Shape      : {df_ball.shape}')
    print(f'\nColumns ({len(df_ball.columns)}):')
    print(df_ball.dtypes.to_string())
    print('\nFirst 5 rows:')
    display(df_ball.head())

    print('\nNull counts:')
    print(df_ball.isnull().sum()[df_ball.isnull().sum() > 0])

In [ ]:
# ── Load corresponding info file ──────────────────────────────────────────────
if RAW_DIR.exists() and ball_files:
    info_path = RAW_DIR / (sample_match.stem + '_info.csv')
    if info_path.exists():
        df_info = pd.read_csv(info_path, header=None)
        print(f'Info file : {info_path.name}')
        print(f'Shape     : {df_info.shape}')
        print()
        display(df_info)

### 2.2 Legacy dataset — ipl_colab.csv (2008–2017)

In [ ]:
colab_path = ROOT / 'ipl_colab.csv'
if colab_path.exists():
    df_colab = pd.read_csv(colab_path)
    print(f'Shape: {df_colab.shape}')
    print(f'Date range: {df_colab["date"].min()} → {df_colab["date"].max()}')
    print(f'\nColumns: {list(df_colab.columns)}')
    display(df_colab.head())
    print(f'\nTeams ({df_colab["batting_team"].nunique()} unique):')
    print(df_colab['batting_team'].value_counts().to_string())
else:
    print('ipl_colab.csv not found')

---
## 3. Preprocessing Pipeline

Raw data → `data/processed/ipl_features.csv`  
Script: `scripts/preprocess_ipl.py`

Agar preprocessed data already exist karta hai (git se aya), to is cell ko run karna zaroori nahi.  
Sirf fresh data pe karna ho tab run karo.

In [ ]:
features_path = PROC_DIR / 'ipl_features.csv'
if features_path.exists():
    print(f'ipl_features.csv already exists ({features_path.stat().st_size / 1e6:.1f} MB)')
    print('Skipping preprocessing. To re-run:')
    print('  python scripts/preprocess_ipl.py')
else:
    print('Processed data not found. Run preprocessing:')
    print('  python scripts/preprocess_ipl.py')
    print('\nThis will take ~5–15 minutes depending on hardware.')

In [ ]:
# ── Optionally run preprocessing inline ───────────────────────────────────────
# Uncomment if you want to run it from this notebook:

# import subprocess, sys
# result = subprocess.run(
#     [sys.executable, str(ROOT / 'scripts' / 'preprocess_ipl.py')],
#     capture_output=True, text=True
# )
# print(result.stdout[-3000:] if result.stdout else '')
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])
print('Inline preprocessing cell ready (uncomment to run)')

---
## 4. Main Training Dataset — ipl_features.csv

In [ ]:
df = pd.read_csv(features_path, low_memory=False)
print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\nSeasons covered: {sorted(df["season"].dropna().unique().astype(int).tolist())}')
display(df.head(3))

In [ ]:
# ── Column groups ─────────────────────────────────────────────────────────────
col_groups = {
    'Match Metadata'   : ['match_id','season','start_date','venue','innings','ball','batting_team','bowling_team'],
    'Players'          : ['striker','non_striker','bowler'],
    'Toss'             : ['toss_winner','toss_decision','toss_winner_batting'],
    'Match State'      : ['runs','wickets','wickets_left','balls_left','legal_balls_bowled','innings_progress','over_number','ball_in_over'],
    'Phase'            : ['phase','is_powerplay','is_middle','is_death'],
    'Momentum'         : ['runs_last_5','wickets_last_5','runs_last_6_balls','wickets_last_6_balls','runs_last_12_balls','wickets_last_12_balls'],
    'Rates & Targets'  : ['current_run_rate','target','target_remaining','required_run_rate','current_minus_required_rr','required_minus_current_rr'],
    'Team Form'        : ['batting_team_form','bowling_team_form','batting_team_venue_form','bowling_team_venue_form','batting_vs_bowling_form'],
    'Player Form'      : ['striker_form_sr','striker_form_avg','bowler_form_econ','bowler_form_strike','batter_vs_bowler_sr','batter_vs_bowler_balls'],
    'Weather'          : ['temperature_c','relative_humidity','wind_kph','dew_risk'],
    'Venue Strength'   : ['venue_avg_first_innings','venue_avg_second_innings','venue_bat_first_win_rate','runs_vs_par'],
    'Targets'          : ['total_runs','winner','win'],
}

print('Feature group summary:')
print(f'{"Group":<22} {"Cols":>4}  Column names')
print('-' * 90)
total = 0
for grp, cols in col_groups.items():
    present = [c for c in cols if c in df.columns]
    total += len(present)
    print(f'{grp:<22} {len(present):>4}  {", ".join(present)}')
print(f'{"TOTAL":<22} {total:>4}')

In [ ]:
# ── Null check ────────────────────────────────────────────────────────────────
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(1)
null_df     = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_df     = null_df[null_df['nulls'] > 0].sort_values('nulls', ascending=False)

print(f'Columns with nulls: {len(null_df)}')
print()
display(null_df)

print('\n(target/required columns are NaN for innings 1 — expected behaviour)')

In [ ]:
# ── Rows per season ───────────────────────────────────────────────────────────
season_counts = df.groupby('season').agg(
    balls=('match_id', 'count'),
    matches=('match_id', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(season_counts['season'].astype(str), season_counts['matches'], color='steelblue')
ax.set_title('IPL Matches per Season in Dataset', fontsize=13)
ax.set_xlabel('Season')
ax.set_ylabel('Matches')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(season_counts.to_string(index=False))

In [ ]:
# ── Team coverage ─────────────────────────────────────────────────────────────
team_balls = df['batting_team'].value_counts()
print(f'Unique teams: {len(team_balls)}')
print()

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(team_balls.index[::-1], team_balls.values[::-1], color='teal')
ax.set_title('Ball records per batting team')
ax.set_xlabel('Balls')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Phase distribution ────────────────────────────────────────────────────────
phase_counts = df['phase'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(phase_counts.values, labels=phase_counts.index, autopct='%1.1f%%',
       colors=['#3b82f6', '#22c55e', '#ef4444'])
ax.set_title('Ball distribution across match phases')
plt.tight_layout()
plt.show()

In [ ]:
# ── First-innings total distribution ─────────────────────────────────────────
first_inn = df[df['innings'] == 1].groupby('match_id')['total_runs'].max().dropna()
print(f'First innings totals — {len(first_inn)} matches')
print(first_inn.describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(first_inn, bins=40, color='darkorange', edgecolor='white')
ax.axvline(first_inn.mean(), color='red', linewidth=2, label=f'Mean={first_inn.mean():.0f}')
ax.set_title('Distribution of IPL 1st Innings Totals')
ax.set_xlabel('Total Runs')
ax.set_ylabel('Matches')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Team Form & Head-to-Head Data

In [ ]:
# ── Team form (last 5 matches) ────────────────────────────────────────────────
tf = pd.read_csv(PROC_DIR / 'team_form_latest.csv')
tf = tf.sort_values('team_form', ascending=False).reset_index(drop=True)
tf['win_pct'] = (tf['team_form'] * 100).round(0).astype(int)

print('Team Form (last 5 matches win rate):')
display(tf[['team', 'win_pct']])

active_teams = [
    'Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans',
    'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians',
    'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad'
]
tf_active = tf[tf['team'].isin(active_teams)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#22c55e' if w >= 60 else '#f59e0b' if w >= 40 else '#ef4444' for w in tf_active['win_pct']]
ax.barh(tf_active['team'], tf_active['win_pct'], color=colors)
ax.axvline(50, color='black', linewidth=1, linestyle='--', label='50%')
ax.set_title('IPL 2026 — Active Teams Recent Form (Last 5 Matches)', fontsize=12)
ax.set_xlabel('Win %')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Head-to-head matchup form ─────────────────────────────────────────────────
mf = pd.read_csv(PROC_DIR / 'matchup_form_latest.csv')
print(f'H2H records: {len(mf)} team-pair combinations')
display(mf.head(10))

# Pivot table for heatmap
mf_active = mf[mf['team'].isin(active_teams) & mf['opponent'].isin(active_teams)].copy()
pivot = mf_active.pivot(index='team', columns='opponent', values='matchup_form')

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([t.split()[-1] for t in pivot.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([t.split()[-1] for t in pivot.index], fontsize=9)
plt.colorbar(im, ax=ax, label='Win rate (row vs col)')
ax.set_title('Head-to-Head Win Rates (Last 5 Meetings)', fontsize=13)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

---
## 6. Venue Statistics

In [ ]:
vs = pd.read_csv(PROC_DIR / 'venue_stats.csv')
print(f'Total venues: {len(vs)}')
display(vs.sort_values('venue_avg_first_innings', ascending=False).head(15).reset_index(drop=True))

In [ ]:
# ── Top venues by avg 1st innings score ───────────────────────────────────────
top15 = vs.sort_values('venue_avg_first_innings', ascending=False).head(15)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Avg 1st innings
axes[0].barh(top15['venue'], top15['venue_avg_first_innings'], color='#3b82f6')
axes[0].set_title('Avg 1st Innings Score')
axes[0].set_xlabel('Runs')
axes[0].invert_yaxis()

# Avg 2nd innings
axes[1].barh(top15['venue'], top15['venue_avg_second_innings'], color='#22c55e')
axes[1].set_title('Avg 2nd Innings Score')
axes[1].set_xlabel('Runs')
axes[1].invert_yaxis()

# Bat-first win rate
colors_wr = ['#22c55e' if w > 0.5 else '#ef4444' for w in top15['venue_bat_first_win_rate']]
axes[2].barh(top15['venue'], top15['venue_bat_first_win_rate'] * 100, color=colors_wr)
axes[2].axvline(50, color='black', linewidth=1, linestyle='--')
axes[2].set_title('Bat-First Win Rate (%)')
axes[2].set_xlabel('Win %')
axes[2].invert_yaxis()

plt.suptitle('Venue Statistics — Top 15 Venues by Average First Innings Score', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Player Statistics

In [ ]:
# ── Batter form ───────────────────────────────────────────────────────────────
bf = pd.read_csv(PROC_DIR / 'batter_form_latest.csv')
print(f'Total batters tracked: {len(bf)}')

# Filter to meaningful sample (min 100 balls)
bf_min = bf[bf['batter_balls'] >= 100].copy()
print(f'Batters with 100+ balls: {len(bf_min)}')

print('\nTop 15 by strike rate (min 100 balls):')
display(bf_min.sort_values('striker_form_sr', ascending=False).head(15)[['batter','batter_balls','batter_runs','striker_form_sr','striker_form_avg']].reset_index(drop=True))

In [ ]:
# ── SR vs Average scatter ─────────────────────────────────────────────────────
bf_min500 = bf[bf['batter_balls'] >= 500].copy()

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    bf_min500['striker_form_avg'],
    bf_min500['striker_form_sr'],
    c=bf_min500['batter_balls'],
    cmap='plasma', alpha=0.7, s=60
)
plt.colorbar(sc, ax=ax, label='Balls faced (career)')

# Label top players
top = bf_min500.nlargest(12, 'striker_form_sr')
for _, row in top.iterrows():
    ax.annotate(row['batter'], (row['striker_form_avg'], row['striker_form_sr']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.axhline(130, color='red', linestyle='--', linewidth=0.8, label='SR = 130')
ax.axvline(25, color='blue', linestyle='--', linewidth=0.8, label='Avg = 25')
ax.set_xlabel('Batting Average', fontsize=11)
ax.set_ylabel('Strike Rate', fontsize=11)
ax.set_title('IPL Batter Form — SR vs Average (career, min 500 balls)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Bowler form ───────────────────────────────────────────────────────────────
bwf = pd.read_csv(PROC_DIR / 'bowler_form_latest.csv')
print(f'Total bowlers tracked: {len(bwf)}')

bwf_min = bwf[bwf['bowler_balls'] >= 120].copy()  # at least 20 overs
print(f'Bowlers with 120+ balls: {len(bwf_min)}')

print('\nTop 15 by economy (min 120 balls):')
display(bwf_min.sort_values('bowler_form_econ').head(15)[['bowler','bowler_balls','bowler_wickets','bowler_form_econ','bowler_form_strike']].reset_index(drop=True))

In [ ]:
# ── Economy vs Strike rate scatter ───────────────────────────────────────────
bwf_min300 = bwf[bwf['bowler_balls'] >= 300].copy()

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    bwf_min300['bowler_form_econ'],
    bwf_min300['bowler_form_strike'],
    c=bwf_min300['bowler_wickets'],
    cmap='YlOrRd', alpha=0.7, s=60
)
plt.colorbar(sc, ax=ax, label='Career wickets')

# Label top wicket-takers
top_wk = bwf_min300.nlargest(10, 'bowler_wickets')
for _, row in top_wk.iterrows():
    ax.annotate(row['bowler'], (row['bowler_form_econ'], row['bowler_form_strike']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.axvline(8.0, color='blue', linestyle='--', linewidth=0.8, label='Economy = 8.0')
ax.set_xlabel('Economy Rate', fontsize=11)
ax.set_ylabel('Strike Rate (balls/wicket)', fontsize=11)
ax.set_title('IPL Bowler Form — Economy vs Strike Rate (career, min 300 balls)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Batter vs Bowler matchups ─────────────────────────────────────────────────
bbf = pd.read_csv(PROC_DIR / 'batter_bowler_form_latest.csv')
print(f'Total batter-bowler matchup pairs: {len(bbf):,}')
display(bbf.head(10))

# Most-faced matchups (most balls)
print('\nTop 10 most-faced matchups:')
display(bbf.sort_values('batter_vs_bowler_balls', ascending=False).head(10).reset_index(drop=True))

---
## 8. Squad Pool — Team Players for 2026

In [ ]:
pp = pd.read_csv(PROC_DIR / 'team_player_pool_2026.csv')
print(f'Total player-team entries: {len(pp)}')
print(f'Teams: {sorted(pp["team"].unique().tolist())}')
print(f'\nPlayers per team:')
print(pp['team'].value_counts().to_string())

In [ ]:
# ── Squad display ─────────────────────────────────────────────────────────────
TEAM_COLORS = {
    'Chennai Super Kings':        '#f5c518',
    'Delhi Capitals':             '#004c97',
    'Gujarat Titans':             '#1b3c5e',
    'Kolkata Knight Riders':      '#3b215d',
    'Lucknow Super Giants':       '#00a3e0',
    'Mumbai Indians':             '#005da0',
    'Punjab Kings':               '#ed1c24',
    'Rajasthan Royals':           '#ea3b94',
    'Royal Challengers Bengaluru':'#c8102e',
    'Sunrisers Hyderabad':        '#f26522',
}

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
for ax, team in zip(axes.flatten(), active_teams):
    squad = pp[pp['team'] == team].sort_values('appearances', ascending=False).head(11)
    color = TEAM_COLORS.get(team, 'gray')
    ax.barh(squad['player'], squad['appearances'], color=color)
    ax.set_title(team.split()[-1] + '/' + team.split()[0][:3], fontsize=9, fontweight='bold')
    ax.tick_params(axis='y', labelsize=7)
    ax.invert_yaxis()
    ax.set_xlabel('Appearances', fontsize=7)

plt.suptitle('IPL 2026 Active Squads — Ball appearances in 2024-2026', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Feature Engineering — Key Transformations

Yeh section explain karta hai kaise raw ball data se features bante hain.

In [ ]:
# ── Run rate & required run rate distribution ─────────────────────────────────
inn2 = df[(df['innings'] == 2) & (df['balls_left'] > 0)].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['current_run_rate'].clip(0, 15), bins=40, color='#3b82f6', edgecolor='white')
axes[0].set_title('Current Run Rate (all innings)')
axes[0].set_xlabel('CRR')

axes[1].hist(inn2['required_run_rate'].clip(0, 24), bins=40, color='#ef4444', edgecolor='white')
axes[1].set_title('Required Run Rate (2nd innings)')
axes[1].set_xlabel('RRR')

delta = inn2['current_minus_required_rr'].clip(-12, 12)
axes[2].hist(delta, bins=40, color='#22c55e', edgecolor='white')
axes[2].axvline(0, color='red', linewidth=1.5)
axes[2].set_title('CRR − RRR\n(+ve = batting ahead)')
axes[2].set_xlabel('CRR − RRR')

plt.tight_layout()
plt.show()

In [ ]:
# ── Win probability by CRR-RRR delta ─────────────────────────────────────────
inn2_valid = inn2[inn2['win'].notna()].copy()
inn2_valid['crr_rrr_bucket'] = pd.cut(
    inn2_valid['current_minus_required_rr'].clip(-10, 10),
    bins=20
)

win_by_delta = inn2_valid.groupby('crr_rrr_bucket', observed=True)['win'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
x_labels = [f'{iv.mid:.1f}' for iv in win_by_delta.index]
ax.bar(range(len(win_by_delta)), win_by_delta.values, color='#7c3aed')
ax.axhline(0.5, color='red', linewidth=1, linestyle='--')
ax.set_xticks(range(len(win_by_delta)))
ax.set_xticklabels(x_labels, rotation=45, fontsize=8)
ax.set_title('Win Rate vs CRR − RRR (2nd innings)', fontsize=12)
ax.set_xlabel('CRR − RRR bucket midpoint')
ax.set_ylabel('Win rate')
plt.tight_layout()
plt.show()

In [ ]:
# ── Team form vs actual win rate ──────────────────────────────────────────────
df_valid = df[df['win'].notna()].copy()
df_valid['form_bucket'] = pd.cut(df_valid['batting_team_form'], bins=5)
form_win = df_valid.groupby('form_bucket', observed=True)['win'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(form_win)), form_win.values, color='darkorange')
ax.axhline(0.5, color='red', linewidth=1, linestyle='--', label='50%')
ax.set_xticks(range(len(form_win)))
ax.set_xticklabels([str(b) for b in form_win.index], rotation=30, fontsize=8)
ax.set_title('Batting Team Form → Actual Win Rate', fontsize=12)
ax.set_ylabel('Win rate')
ax.legend()
plt.tight_layout()
plt.show()

print('Higher form → higher win rate — confirms feature validity')

---
## 10. Dataset Summary Card

In [ ]:
print('=' * 60)
print(' IPL PREDICTION PROJECT — DATA SUMMARY')
print('=' * 60)

files = {
    'ipl_features.csv (main training)': features_path,
    'team_form_latest.csv':              PROC_DIR / 'team_form_latest.csv',
    'team_venue_form_latest.csv':        PROC_DIR / 'team_venue_form_latest.csv',
    'matchup_form_latest.csv':           PROC_DIR / 'matchup_form_latest.csv',
    'venue_stats.csv':                   PROC_DIR / 'venue_stats.csv',
    'batter_form_latest.csv':            PROC_DIR / 'batter_form_latest.csv',
    'bowler_form_latest.csv':            PROC_DIR / 'bowler_form_latest.csv',
    'batter_bowler_form_latest.csv':     PROC_DIR / 'batter_bowler_form_latest.csv',
    'team_player_pool_2026.csv':         PROC_DIR / 'team_player_pool_2026.csv',
}

print(f'\n{"File":<40} {"Rows":>8} {"Cols":>5} {"Size":>10}')
print('-' * 68)
for name, path in files.items():
    if path.exists():
        tmp = pd.read_csv(path, nrows=0)
        n   = sum(1 for _ in open(path)) - 1
        sz  = path.stat().st_size
        sz_str = f'{sz/1e6:.1f} MB' if sz >= 1e6 else f'{sz/1e3:.0f} KB'
        print(f'{name:<40} {n:>8,} {len(tmp.columns):>5} {sz_str:>10}')
    else:
        print(f'{name:<40} {"MISSING":>8}')

print('\n' + '=' * 60)
print(f'  Raw match files  : {len(list(RAW_DIR.glob("*.csv"))) if RAW_DIR.exists() else "N/A"}')
print(f'  Seasons          : IPL 2007–2026')
print(f'  Active teams     : 10')
print(f'  Total ML features: 61 (10 categorical + 51 numeric)')
print('=' * 60)

---
## Next Steps

Is notebook ke baad:

1. **Model training (CPU):** `python scripts/train_models.py`
2. **Model training (GPU):** `python scripts/train_gpu_best.py`
3. **Pre-match model:** `python scripts/train_pre_match.py`
4. **Flask web app:** `python web_app.py`
5. **Streamlit predictor:** `streamlit run streamlit_app.py`

Detailed docs: `docs/DATA_SUMMARY.md` aur `docs/ARCHITECTURE.md`